# Cascaded-SolarNet: test + train on Colab

Clones the repo, runs the test suite, downloads InfraredSolarModules, trains the cascade and baselines, and evaluates end to end.

**Before you start:** Runtime -> Change runtime type -> **T4 GPU**.

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 1. Clone and install

In [ ]:
%cd /content
!rm -rf cascaded-cnn
!git clone https://github.com/techwack/cascaded-cnn
%cd /content/cascaded-cnn
# torch is already installed on Colab; install the rest
!pip install -q numpy scikit-learn pillow pyyaml pytest
!pip install -q -e . --no-deps

## 2. Run the tests (synthetic data, no download needed)

In [ ]:
!python -m pytest -q

## 3. Download the dataset
Fetches InfraredSolarModules from the original RaptorMaps GitHub repo (CC BY 4.0). No Kaggle token needed.

In [ ]:
!bash scripts/download_data.sh
!python scripts/make_splits.py

## 4. Train
Each node trains on ground-truth routing. Early stopping is on val macro-F1 (patience 20, up to 100 epochs).

In [ ]:
%cd /content/cascaded-cnn
!python -m solarnet.train --node all

In [ ]:
%cd /content/cascaded-cnn
!python -m solarnet.train --node flat
!python baselines/random_forest.py

## 5. Evaluate the whole cascade on the test split

In [ ]:
%cd /content/cascaded-cnn
!python -m solarnet.evaluate --flat

In [ ]:
%cd /content/cascaded-cnn
import json
r = json.load(open("runs/results.json"))
print(json.dumps(r, indent=2)[:6000])

## 6. Save results
Downloads the logs and metrics (not the large checkpoints) so you can commit them or paste numbers into the report.

In [ ]:
%cd /content/cascaded-cnn
!zip -qr results.zip runs -x "*.pt"
from google.colab import files
files.download("results.zip")